In [52]:
import numpy as np
import pandas as pd
import json

# -------------------------- 定义常量 --------------------------
hidden_states = ['CAPEC'+str(i) for i in range(1,500)]

event_types = ['CAR'+str(i) for i in range(1,1000)]

def generate_transition_probs():

    transition_probs = {}
    for hidden_state in hidden_states:
        # 从hidden_states中随机选择5个状态
        selected_states = np.random.choice(hidden_states, size=5, replace=False)
        
        # 生成5个随机概率值,确保和为1
        probs = np.random.dirichlet(np.ones(5))
        
        # 构建转移概率字典
        state_dict = {}
        for state, prob in zip(selected_states, probs):
            state_dict[state] = prob
        
        transition_probs[hidden_state] = state_dict
    
    return transition_probs

def generate_emission_probs():
    emission_probs = {}
    for hidden_state in hidden_states:
        selected_events = np.random.choice(event_types, size=5, replace=False)
        
        # 生成5个随机概率值,确保和为1
        probs = np.random.dirichlet(np.ones(5))
        
        # 构建观测概率字典
        event_dict = {}
        for event, prob in zip(selected_events, probs):
            event_dict[event] = prob
        emission_probs[hidden_state] = event_dict
    return emission_probs

transition_probs = generate_transition_probs()
emission_probs = generate_emission_probs()
#----------------------------保存转移概率和观测概率----------------------------
model_params = {
    'transition_probs': transition_probs,
    'emission_probs': emission_probs
}
with open('probs_matrix.json', 'w') as f:
    json.dump(model_params, f)

# -------------------------- 生成模拟数据 --------------------------
def generate_attack_sequence(sq_length_start=5,sq_length_end=10):
    """生成单个攻击序列"""
        
    sequence_length = np.random.randint(sq_length_start, sq_length_end)
    states = []
    events = []
    
    current_state = 'CAPEC1'
    
    for _ in range(sequence_length):
        possible_events = list(emission_probs[current_state].keys())
        event_probs = list(emission_probs[current_state].values())
        event = np.random.choice(possible_events, p=event_probs)
        events.append(event)
        states.append(current_state)
        
        possible_next_states = list(transition_probs[current_state].keys())
        state_probs = list(transition_probs[current_state].values())
        current_state = np.random.choice(possible_next_states, p=state_probs)
    
    return states, events

def generate_dataset(num_sequences=1000,sq_length_start=5,sq_length_end=10):
    sequences = []
    for _ in range(num_sequences):
        states, events = generate_attack_sequence(sq_length_start,sq_length_end)
        sequence = {
            'states': states,
            'events': events
        }
        sequences.append(sequence)
    return sequences

# -------------------------- 保存和加载数据 --------------------------
def save_dataset(sequences, filename):
    with open(filename, 'w') as f:
        json.dump(sequences, f)

# -------------------  生成数据 -------------------
# 生成数据
print("生成数据集...")
dataset = generate_dataset(num_sequences=1500,sq_length_start=5,sq_length_end=10)

# 保存数据
save_dataset(dataset, 'attack_sequences.json')

生成数据集...


In [56]:
import numpy as np
import pandas as pd
import json
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import DBNInference
from collections import defaultdict

# -------------------------- 定义常量 --------------------------
# -------------------------- 定义常量 --------------------------
hidden_states = ['CAPEC'+ str(i) for i in range(1,500)]
event_types = ['CAR'+ str(i) for i in range(1,1000)]

event_to_code = {event: idx for idx, event in enumerate(event_types)}
state_to_code = {state: idx for idx, state in enumerate(hidden_states)}
# -------------------------- 参数学习函数 --------------------------
class DBNParameterLearner:
    def __init__(self, n_states, n_events):
        self.n_states = n_states
        self.n_events = n_events
        
        # 初始化计数器
        self.state_transitions = np.zeros((n_states, n_states))
        self.state_emissions = np.zeros((n_events, n_states))  # 注意这里交换了维度顺序
        self.initial_states = np.zeros(n_states)
        
    def count_transitions(self, sequences):
        """统计状态转移和观测事件"""
        for sequence in sequences:
            states = [state_to_code[s] for s in sequence['states']]
            events = [event_to_code[e] for e in sequence['events']]
            
            # 统计初始状态
            self.initial_states[states[0]] += 1
            
            # 统计状态转移
            for i in range(len(states)-1):
                current_state = states[i]
                next_state = states[i+1]
                self.state_transitions[current_state][next_state] += 1
            
            # 统计观测事件
            for i in range(len(states)):
                state = states[i]
                event = events[i]
                self.state_emissions[event][state] += 1  # 注意这里交换了索引顺序
    
    def estimate_parameters(self, smoothing=0.1):
        """估计转移概率和观测概率"""
        # 添加平滑处理
        self.state_transitions += smoothing
        self.state_emissions += smoothing
        self.initial_states += smoothing
        
        # 计算概率
        # 确保转移矩阵按行归一化
        transition_matrix = self.state_transitions / self.state_transitions.sum(axis=1, keepdims=True)
        
        # 确保观测矩阵按列归一化
        emission_matrix = self.state_emissions / self.state_emissions.sum(axis=0, keepdims=True)
        
        # 确保初始概率和为1
        initial_probs = self.initial_states / self.initial_states.sum()
        
        return transition_matrix, emission_matrix, initial_probs

    def learn_parameters(self, sequences):
        """学习模型参数"""
        self.count_transitions(sequences)
        return self.estimate_parameters()

# -------------------------- 创建DBN模型 --------------------------
def create_dbn_model(transition_matrix, emission_matrix, initial_probs):
    model = DBN()
    
    # 添加节点和边
    model.add_nodes_from(["Hidden", "Event"])
    model.add_edge(("Hidden", 0), ("Hidden", 1))
    model.add_edge(("Hidden", 0), ("Event", 0))
    model.add_edge(("Hidden", 1), ("Event", 1))
    
    # 设置CPD
    init_state = TabularCPD(
        variable=("Hidden", 0),
        variable_card=len(hidden_states),
        values=initial_probs.reshape(-1, 1)
    )
    
    trans_model = TabularCPD(
        variable=("Hidden", 1),
        variable_card=len(hidden_states),
        evidence=[("Hidden", 0)],
        evidence_card=[len(hidden_states)],
        values=transition_matrix.T  # 转置以符合pgmpy的要求
    )
    
    obs_model_0 = TabularCPD(
        variable=("Event", 0),
        variable_card=len(event_types),
        evidence=[("Hidden", 0)],
        evidence_card=[len(hidden_states)],
        values=emission_matrix
    )
    
    obs_model_1 = TabularCPD(
        variable=("Event", 1),
        variable_card=len(event_types),
        evidence=[("Hidden", 1)],
        evidence_card=[len(hidden_states)],
        values=emission_matrix
    )
    
    model.add_cpds(init_state, trans_model, obs_model_0, obs_model_1)
    
    # 验证概率和是否为1
    print("\n验证概率矩阵:")
    print("转移矩阵行和:", transition_matrix.sum(axis=1))
    print("观测矩阵列和:", emission_matrix.sum(axis=0))
    print("初始概率和:", initial_probs.sum())
    
    model.check_model()
    return model

# -------------------------- 评估函数 --------------------------
def evaluate_model(model, test_sequences):
    infer = DBNInference(model)
    correct = 0
    total = 0
    class_correct = defaultdict(int)
    class_total = defaultdict(int)
    
    for sequence in test_sequences:
        events = sequence['events']
        states = sequence['states']
        
        for i in range(len(events)-1):
            evidence = {
                ("Event", 0): event_to_code[events[i]],
                ("Event", 1): event_to_code[events[i+1]]
            }
            
            # 预测隐状态
            forward_state = infer.forward_inference([("Hidden", 1)], evidence)
            predicted_state_idx = np.argmax(forward_state[("Hidden", 1)].values)
            predicted_state = hidden_states[predicted_state_idx]
            
            # 比较预测结果
            actual_state = states[i+1]
            if predicted_state == actual_state:
                correct += 1
                class_correct[actual_state] += 1
            total += 1
            class_total[actual_state] += 1
    
    # 计算整体准确率和每个类别的准确率
    accuracy = (correct / total) * 100
    class_accuracy = {state: (class_correct[state] / class_total[state] * 100) 
                     if class_total[state] > 0 else 0 
                     for state in hidden_states}
    
    return accuracy, class_accuracy

# -------------------------- 主程序 --------------------------
if __name__ == "__main__":
    # 加载数据
    print("加载数据...")
    with open('attack_sequences.json', 'r') as f:
        sequences = json.load(f)
    
    # 划分训练集和测试集
    split_idx = int(len(sequences) * 0.8)
    train_sequences = sequences[:split_idx]
    test_sequences = sequences[split_idx:]
    
    # 学习参数
    print("学习模型参数...")
    learner = DBNParameterLearner(len(hidden_states), len(event_types))
    transition_matrix, emission_matrix, initial_probs = learner.learn_parameters(train_sequences)
    
    # 创建模型
    print("创建DBN模型...")
    model = create_dbn_model(transition_matrix, emission_matrix, initial_probs)
    
    # 保存模型参数而不是模型本身
    model_params = {
        'transition_matrix': transition_matrix.tolist(),
        'emission_matrix': emission_matrix.tolist(), 
        'initial_probs': initial_probs.tolist()
    }
    print("保存模型参数...")
    with open('./save/dbn_model_params.json', 'w') as f:
        json.dump(model_params, f)

加载数据...
学习模型参数...
创建DBN模型...

验证概率矩阵:
转移矩阵行和: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1

In [58]:

# 加载模型参数
print("加载模型参数...")
with open('./save/dbn_model_params.json', 'r') as f:
    model_params = json.load(f)

# 从参数创建模型
print("创建DBN模型...")
transition_matrix = np.array(model_params['transition_matrix'])
emission_matrix = np.array(model_params['emission_matrix'])
initial_probs = np.array(model_params['initial_probs'])
model = create_dbn_model(transition_matrix, emission_matrix, initial_probs)

# 加载数据
print("加载数据...")
with open('attack_sequences.json', 'r') as f:
    sequences = json.load(f)

# 划分训练集和测试集
split_idx = int(len(sequences) * 0.05)
train_sequences = sequences[:split_idx]
test_sequences = sequences[split_idx:]



# 评估模型
print("评估模型...")
accuracy, class_accuracy = evaluate_model(model, test_sequences)

print(f"\n整体准确率: {accuracy:.2f}%")
print("\n各状态准确率:")
for state, acc in class_accuracy.items():
    print(f"{state}: {acc:.2f}%")

# 保存学习到的参数
learned_params = {
    'transition_matrix': transition_matrix.tolist(),
    'emission_matrix': emission_matrix.tolist(),
    'initial_probabilities': initial_probs.tolist(),
    'accuracy': accuracy,
    'class_accuracy': class_accuracy
}

with open('learned_dbn_params.json', 'w') as f:
    json.dump(learned_params, f, indent=4)

# 示例预测
print("\n示例预测:")
sample_sequence = test_sequences[0]
evidence = {
    ("Event", 0): event_to_code[sample_sequence['events'][0]],
    ("Event", 1): event_to_code[sample_sequence['events'][1]]
}

infer = DBNInference(model)
forward_state = infer.forward_inference([("Hidden", 1)], evidence)

print(f"输入序列: {sample_sequence['events'][:2]}")
print("预测的隐状态概率:")
for i, state in enumerate(hidden_states):
    prob = forward_state[("Hidden", 1)].values[i]
    print(f"{state}: {prob:.3f}")

加载模型参数...


创建DBN模型...

验证概率矩阵:
转移矩阵行和: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1

KeyboardInterrupt: 